In [1]:
!pip install -q --upgrade datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.0 MB/s eta 0:00:00


In [2]:
!nvidia-smi

Wed Apr 29 03:11:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import userdata
import warnings
import torch
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

warnings.filterwarnings("ignore")


hf_token = userdata.get('HF_TOKEN')
login(hf_token)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# Generic Pipeline syntax:

Create Pipeline
```
my_pipeline = pipeline(task, model=xx, device=yy)
```

Inference:
```
my_pipeline(input)
```

# Sentiment Analysis

In [4]:
simple_sentiment_analysis_pipeline = pipeline("sentiment-analysis", device="cuda")
result = simple_sentiment_analysis_pipeline("This pizza is bland")
result

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9997900128364563}]

In [5]:
result = simple_sentiment_analysis_pipeline("I should be more excited to have this pizza")
result

[{'label': 'POSITIVE', 'score': 0.9913839101791382}]

In [6]:
bert_sentiment_classifier_pipeline = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = bert_sentiment_classifier_pipeline("I should be more excited to have this pizza")
result

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': '3 stars', 'score': 0.3735803961753845}]

# NER

In [7]:
ner_pipeline = pipeline("ner", device="cuda")
result = ner_pipeline("The task is to learn Huggingface Pipelines")
for entity in result:
  print(entity)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

{'entity': 'I-MISC', 'score': np.float32(0.96381223), 'index': 6, 'word': 'Hu', 'start': 21, 'end': 23}
{'entity': 'I-MISC', 'score': np.float32(0.49328753), 'index': 7, 'word': '##gging', 'start': 23, 'end': 28}
{'entity': 'I-MISC', 'score': np.float32(0.8294), 'index': 8, 'word': '##face', 'start': 28, 'end': 32}
{'entity': 'I-MISC', 'score': np.float32(0.71739286), 'index': 9, 'word': 'Pi', 'start': 33, 'end': 35}
{'entity': 'I-ORG', 'score': np.float32(0.39039585), 'index': 10, 'word': '##pel', 'start': 35, 'end': 38}
{'entity': 'I-MISC', 'score': np.float32(0.7455475), 'index': 11, 'word': '##ines', 'start': 38, 'end': 42}


# Question Answering with Context

In [8]:
question = "What are Huggingface Pipelines?"
context = "Pipelines are high level APIs for inference of LLMs with common tasks."

question_answerer = pipeline("question-answering", device="cuda")
result = question_answerer(question=question, context=context)

result

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'score': 0.5682854652404785,
 'start': 14,
 'end': 29,
 'answer': 'high level APIs'}

# Text Summarization

In [ ]:
summarizer = pipeline("summarizer", device="cuda")

# Translation

In [18]:
translator = pipeline("translation_en_to_fr", device="cuda")
result = translator("Translate this text to French")
result

KeyError: 'translation'

# Classification

In [9]:
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Huggingface transformer library is amazing", candidate_labels=["technology", "sports", "politics"])
result

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'Huggingface transformer library is amazing',
 'labels': ['technology', 'sports', 'politics'],
 'scores': [0.983738124370575, 0.010023941285908222, 0.006237946450710297]}